In [ ]:
# compute_homography.py
import cv2
import numpy as np
import glob, os

IMG_DIR = "calibration/images"
OUT_DIR = "calibration/output"
CHECKERBOARD = (9, 6)
SQUARE_SIZE_M = 0.02

os.makedirs(OUT_DIR, exist_ok=True)

def compute_homography_from_image(fname):
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (CHECKERBOARD[0], CHECKERBOARD[1]), None)
    if not ret:
        raise RuntimeError(f"Chessboard not found in {fname}")
    corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1),
                                criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-6))
    # Pixel coordinates (u,v)
    pts_src = corners2.reshape(-1, 2)

    # Physical shelf coordinates: choose coordinate system on the checkerboard plane
    # We'll map the chessboard inner corners to plane coordinates (X,Y) in meters
    objp = np.zeros((CHECKERBOARD[0]*CHECKERBOARD[1], 2), dtype=np.float32)
    objp[:, 0], objp[:, 1] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]]
    objp *= SQUARE_SIZE_M  # scale to meters

    # compute homography mapping pixels -> phys XY
    H, status = cv2.findHomography(pts_src, objp, 0)
    return H, img

if __name__ == "__main__":
    # pick one image where the checkerboard lies in the shelf plane and is well visible
    images = glob.glob(os.path.join(IMG_DIR, "*.jpg"))
    if not images:
        print("No images to compute homography from.")
        exit(1)
    # choose first or ask user to specify - here we pick the first that finds chessboard
    for fname in images:
        try:
            H, img = compute_homography_from_image(fname)
            np.save(os.path.join(OUT_DIR, "homography_pixel_to_shelf.npy"), H)
            print("Saved homography to", os.path.join(OUT_DIR, "homography_pixel_to_shelf.npy"))
            break
        except Exception as e:
            print("Skipping", fname, ":", e)
    else:
        print("No suitable image found with chessboard.")
